# Langchain Middleware And Runtime State
<img src="../../assets/runtime_params.png" width="1000" height="400">
<img src="../../assets/agentic_components.png" width="1000" height="400">


In [13]:
import os
from dotenv import load_dotenv
from typing import Literal
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field
from langchain.messages import HumanMessage
from langchain.tools import tool
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import dynamic_prompt, HumanInTheLoopMiddleware, ToolCallRequest
from langgraph.types import Command

from rich import print
import sys
sys.path.append('..')
from utils.helper import pretty_print_agent_output, print_messages

load_dotenv()

True

In [3]:
if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY environment variable is not set.")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

model = ChatOpenAI(model="gpt-5-nano")

## Dynamic Prompting
* Dynamically modify the system prompt at runtime to inject context, user-specific instructions, or other information before each model call.
* Use the `system_message` field on `ModelRequest` to read and modify the system prompt.
* It contains a `SystemMessage` object.

In [13]:
class UserContext(BaseModel):
    user_name: str = Field(..., description="The unique identifier for the user.")
    user_role: Literal["developer", "manager"] = Field(..., description="The role of the user, which can be either 'developer' or 'manager'.")

In [22]:
@dynamic_prompt
def personalize_the_prompt(request):
    user_role = request.runtime.context.user_role
    system_prompt=''
    if user_role == "developer":
        system_prompt = """
        You are an expert developer.

        Give technical answers with code examples.
        Assume familiarity with programming concepts.
        """
    if user_role == "manager":
        system_prompt = """
        You are assisting an engineering manager.

        Focus on architecture, risks, trade-offs,
        timelines, and business impact.
        Avoid unnecessary implementation details.
        """
    print("\n========== SYSTEM PROMPT ==========")
    print(system_prompt)
    print("===================================\n")
    return system_prompt

In [23]:
dynamic_system_prompt_agent = create_agent(
    model="openai:gpt-5-nano",
    tools=[],
    middleware=[personalize_the_prompt],
    context_schema=UserContext,
)

In [24]:
result = dynamic_system_prompt_agent.invoke({"messages": [HumanMessage(content="Write a Python function to print hello world")]},
                                            context={"user_name": "Shivam", "user_role": "developer"})


========== SYSTEM PROMPT ==========

        You are an expert developer.

        Give technical answers with code examples.
        Assume familiarity with programming concepts.
        



In [25]:
result = dynamic_system_prompt_agent.invoke({"messages": [HumanMessage(content="Write a sentence about the importance of code reviews")]},
                                            context={"user_name": "Satyam", "user_role": "manager"})


========== SYSTEM PROMPT ==========

        You are assisting an engineering manager.

        Focus on architecture, risks, trade-offs,
        timelines, and business impact.
        Avoid unnecessary implementation details.
        



# Human-in-the-loop
* The Human-in-the-Loop (HITL) middleware lets you add human oversight to agent tool calls. 
* It does this by checking each tool call against a configurable policy. 
* If intervention is needed, the middleware issues an interrupt that halts execution. 
* The graph state is saved using LangGraph’s persistence layer, so execution can pause safely and resume later.
* A human decision then determines what happens next:

**Human Actions**
1. **approve**: Execute the tool with the original arguments as proposed by the agent.
2. **edit**: Modify the tool arguments before execution.
3. **reject**: Skip executing this tool call entirely and return rejection feedback to the agent.
4. **respond**: Return the human’s message directly as a synthetic tool result, skipping execution. Use respond only when the human is acting as the tool.

#### Configuring interrupts
* `interrupt_on` configure agent with a mapping of tool actions to the decision types that are allowed for each action.
* The middleware will interrupt execution when a tool call matches an action in the mapping.
* This needs to configure a checkpointer to persist the graph state across interrupts.

**Tool definitions**
1. read_file(path): reads the contents of a file
2. write_file(path, content): writes a file
3. execute_database_query(query): simulates a database query execution


In [ ]:
@tool
def read_file(path: str) -> str:
    """Read content from a file at the specified path."""
    with open(path, "r") as f:
        return f.read()

@tool
def write_file(path: str, content: str) -> str:
    """Write content to a file at the specified path."""
    with open(path, "w") as f:
        f.write(content)
    return f"Content written to {path}"

@tool
def execute_database_query(query: str) -> str:
    """
    Use this tool to execute a SQL query against the sales database.
    This tool is intended to organize or analyze sales data, including questions about sales
    schema of the sales database.
    sales(sales_id, product_id, quantity, price, sale_date)
    Args:
        query (str): A SQL query in SQLite3 dialect.
    """
    return f"Executed query: {query}"

#### Conditional interrupts
* By default, every tool call listed in `interrupt_on` pauses for review. 
* To pause only some calls, add `when` predicate, the predicate receives ToolCallRequest and returns True to interrupt or False to auto-approve.

**Defines two guard functions**
1. `writes_outside_workspace(request)`: interrupts if a write is outside ./ (the allowed workspace)
2. `is_dml_query(request)`: interrupts if a SQL query is not a read-only SELECT

In [51]:
def writes_outside_workspace(request: ToolCallRequest) -> bool:
    """Pause writes to paths outside the workspace directory."""
    path = request.tool_call["args"].get("path", "")
    return not path.startswith("./")

def is_dml_query(request: ToolCallRequest) -> bool:
    """Pause SQL that isn't a read-only SELECT."""
    query = request.tool_call["args"].get("query", "")
    return not query.lstrip().upper().startswith("SELECT")

**Create Agent**
* The agent is configured to pause execution when the model tries to 
    1. Write outside the workspace
    2. Run dangerous SQL

In [ ]:
agent = create_agent(
    model="gpt-5-mini",
    tools=[read_file, write_file, execute_database_query],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "write_file": {
                    "allowed_decisions": ["approve", "edit", "reject", "respond"],
                    "when": writes_outside_workspace,
                },
                "execute_database_query": {
                    "allowed_decisions": ["approve", "reject"],
                    "when": is_dml_query,
                },
                "read_file": False # Always allow reading files without interruption
            },
            description_prefix="Tool execution pending approval",
        ),
    ],
    checkpointer=InMemorySaver(),
    system_prompt="""You are a helpful data organization assistant, use the available tools to assist 
    the user in managing and organizing their data in databases and file-systems effectively.
    """,
)

### Responding to interrupts
* When you invoke the agent, it runs until it either completes or an interrupt is raised. 
* With **version="v2"**, the result is a `GraphOutput` with an `interrupts` and `value` attributes containing the actions that require review. 
* We can then present these actions to a reviewer and resume execution once decisions are provided.

**Human-in-the-loop leverages LangGraph's persistence layer**
* Need a thread ID to associate the execution with a conversation thread, so the conversation can be paused and resumed.

**First interrupt example: delete old records**
* Calls the agent with a request to delete records older than 2025-08-01 from the sales table.
* The model tries to call execute_database_query with a DELETE-style SQL

In [36]:
config = {"configurable": {"thread_id": "user_12"}}

result = agent.invoke({"messages": [HumanMessage(content="Delete records from the sales table that are older than 1-Aug-2025 sale_date")]},
        config=config, version="v2",
)

**Messages stored in the result state**
* The agent first checks how many records match the deletion condition by running a `SELECT COUNT(*)` query.
* This is a read-only operation, so it is allowed and does not trigger the HITL interrupt.
* After that, the agent attempts the actual `DELETE FROM sales ...` statement.
* Because this is a data-modifying action, it matches the interrupt policy `is_dml_query` and pauses for human approval.

In [39]:
print_messages(result.value)

HumanMessage: Delete records from the sales table that are older than 1-Aug-2025 sale_date

================================================================================

Agent: AIMessage with tool calls:

Tool Call 1 Name: execute_database_query
Tool Call 1 Args: {'query': "SELECT COUNT(*) AS to_delete FROM sales WHERE sale_date < '2025-08-01';"}

================================================================================

ToolMessage: Executed query: SELECT COUNT(*) AS to_delete FROM sales WHERE sale_date < '2025-08-01';

================================================================================

Agent: AIMessage with tool calls:

Tool Call 1 Name: execute_database_query
Tool Call 1 Args: {'query': "DELETE FROM sales WHERE sale_date < '2025-08-01';"}

================================================================================

**Display the list of pending human decisions**
* Shows exactly which tool call is waiting for approval and which decisions are allowed
    * Tool name: execute_database_query
    * arguments: { "query": "DELETE FROM sales WHERE ..." }
    * allowed_decisions: ["approve", "reject"]
    * interrupt reason: query is not a read-only SELECT

In [37]:
print(result.interrupts)

(
    Interrupt(
        value={
            'action_requests': [
                {
                    'name': 'execute_database_query',
                    'args': {'query': "DELETE FROM sales WHERE sale_date < '2025-08-01';"},
                    'description': 'Tool execution pending approval\n\nTool: execute_database_query\nArgs: 
{\'query\': "DELETE FROM sales WHERE sale_date < \'2025-08-01\';"}'
                }
            ],
            'review_configs': [
                {'action_name': 'execute_database_query', 'allowed_decisions': ['approve', 'reject']}
            ]
        },
        id='e7e1ae21a651d96ba86351a66a54d427'
    ),
)

**Resume with reject**
* The tool call is not executed.
* The agent continues with the rejection feedback and returns a final assistant message explaining it did not delete the records.


In [ ]:
reject_response = agent.invoke(
    Command(
        resume={"decisions": [{"type": "reject", "reason": "No need to delete old records"}]}
    ),
    config=config, # Same thread ID to resume the paused conversation
    version="v2",
)

In [43]:
print(reject_response.value['messages'][-1].content)

I can delete those rows, but I won't run the destructive DELETE until you confirm. Options — tell me which you'd 
like me to do:

1) Preview how many rows would be deleted (recommended first)
   SQL:
   SELECT COUNT(*) AS to_delete FROM sales WHERE sale_date < '2025-08-01';

2) Preview the rows (sample)
   SQL:
   SELECT * FROM sales WHERE sale_date < '2025-08-01' LIMIT 100;

3) Archive then delete (safe: copy rows to an archive table first)
   Example sequence:
   -- create empty archive table with same columns
   CREATE TABLE IF NOT EXISTS sales_archive AS SELECT * FROM sales WHERE 0;
   -- copy the rows to archive
   INSERT INTO sales_archive SELECT * FROM sales WHERE sale_date < '2025-08-01';
   -- delete the original rows
   DELETE FROM sales WHERE sale_date < '2025-08-01';

4) Direct delete (I will run it if you confirm)
   Best run inside a transaction:
   BEGIN TRANSACTION;
   DELETE FROM sales WHERE sale_date < '2025-08-01';
   COMMIT;

5) Delete in batches (if the table is large, to avoid long locks)
   Example (repeat until no rows left):
   DELETE FROM sales WHERE rowid IN (SELECT rowid FROM sales WHERE sale_date < '2025-08-01' LIMIT 10000);

Which would you like me to do now? If you want me to proceed with deletion, say “Delete now” or if you want a 
count/preview first say “Show count” or “Show preview”.

**Second interrupt example**
* Tries to create a Python file with content print('Hello, World!') using write_file.
* Uses a new thread id: user_123

In [44]:
config = {"configurable": {"thread_id": "user_123"}}

result = agent.invoke({"messages": [HumanMessage(content="create and write a python file hello.py, with content print('Hello, World!')")]},
        config=config, version="v2",
)

**Messages from the second interrupted run**
* The model decides to call write_file with path as a plain relative file path, such as hello.py or some untrusted path without ./.

In [47]:
print_messages(result)

HumanMessage: create and write a python file hello.py, with content print('Hello, World!')

================================================================================

Agent: AIMessage with tool calls:

Tool Call 1 Name: write_file
Tool Call 1 Args: {'path': 'hello.py', 'content': "print('Hello, World!')\n"}

================================================================================

**Interrupt details for the file write**
* It confirms the trigger: `writes_outside_workspace` returned True.
* Shows which allowed actions are available for the file write.

In [45]:
print(result.interrupts)

(
    Interrupt(
        value={
            'action_requests': [
                {
                    'name': 'write_file',
                    'args': {'path': 'hello.py', 'content': "print('Hello, World!')\n"},
                    'description': 'Tool execution pending approval\n\nTool: write_file\nArgs: {\'path\': 
\'hello.py\', \'content\': "print(\'Hello, World!\')\\n"}'
                }
            ],
            'review_configs': [
                {'action_name': 'write_file', 'allowed_decisions': ['approve', 'edit', 'reject', 'respond']}
            ]
        },
        id='456ec8650163cac92ec65a33b7b57dae'
    ),
)

**Resume with edit decision**
* Human edits the action before execution.
* Lets changes the path from `hello.py` to `./hello.py`and keeps the content.

In [ ]:
resumed = agent.invoke(Command(resume={"decisions": 
                  [{"type": "edit", "edited_action":
                      {"name": "write_file", 
                       "args": {'path': './hello.py', 'content': "print('Hello, World!')\n"}
                       }
                  }]}), config=config, version="v2")

**Assistant response after the edited call is accepted**
* Tool will be called successfully and new `hello.py` file will be created.
* Repond with successfull message

In [50]:
print(resumed.value['messages'][-1].content)

I created the file hello.py with the requested content:

print('Hello, World!')

Files written:
- ./hello.py

If you want, I can run it and show the output, or save a different filename/location.